# Modeling and solving search problems

Consider the following river crossing puzzle:

## Jealous Husbands Problem with an Island

There are n couples: $(𝐻_1,𝑊_1)$, $(𝐻_2,𝑊_2)$, …, $(𝐻_𝑛,𝑊_𝑛)$  where $𝐻_𝑖$ is husband $𝑖$ and $𝑊_𝑖$ is wife $𝑖$.

They need to cross from the left bank of a river to the right bank.

There is a boat that can carry at most two people at a time.

Important: There is an island midway between the left and right banks. The boat can stop at the island.

Rules:

1. Jealousy constraint: No wife may ever be in the presence of another husband (on any shore or the island or in the boat) unless her own husband is also present. Otherwise, fights (and failures) occur.

2. The boat needs at least one person to operate (no remote-controlled trips).

3. The boat can travel between:

            Left bank ↔ Island

            Island ↔ Right bank

4. All people, including the boat, start at the left bank.

Goal: get everyone (husbands and wives) safely to the right bank, obeying all the rules.


## Tasks:

1. Put the problem in a form of a search tree: root, nodes, solution and rules for constructing the childrens.
2. Solve it usinh a classic DFS
3. Solve it using a classic Back-tracking.
4. Solve it using the A* method.  

In [17]:
import itertools
import heapq

people = ["H1", "W1", "H2", "W2"]
husbands = {"H1", "H2"}
wives = {"W1", "W2"}
couples = {"H1": "W1", "H2": "W2", "W1": "H1", "W2": "H2"}

def is_valid(group):
    group = set(group)
    for w in wives:
        if w in group:
            h = couples[w]
            if h not in group and any(hx in group for hx in husbands if hx != h):
                return False
    return True

def is_goal(state):
    _, _, right, _ = state
    return set(right) == set(people)

def generate_children(state):
    left, island, right, boat = state
    loc_map = {"left": left, "island": island, "right": right}
    neighbors = {"left": ["island"], "island": ["left", "right"], "right": ["island"]}
    children = []

    for target in neighbors[boat]:
        source = sorted(loc_map[boat]) 
        dest = loc_map[target]
        for move in sorted(itertools.chain(itertools.combinations(source, 1), itertools.combinations(source, 2))):
            new_source = set(source) - set(move)
            new_dest = set(dest) | set(move)
            if not all(is_valid(g) for g in [new_source, new_dest, move]):
                continue
            new_state = {
                "left": set(left),
                "island": set(island),
                "right": set(right),
            }
            new_state[boat] = new_source
            new_state[target] = new_dest
            children.append((
                frozenset(new_state["left"]),
                frozenset(new_state["island"]),
                frozenset(new_state["right"]),
                target
            ))
    return sorted(children)

def heuristic(state):
    left, island, right, boat = state
    return (len(left) + len(island)) // 2 + (boat != "left")

def dfs(state, path, visited):
    if is_goal(state):
        return path + [state]
    visited.add(state)
    for child in generate_children(state):
        if child not in visited:
            result = dfs(child, path + [child], visited)
            if result:
                return result
    return None

def backtrack(state, path, visited):
    if is_goal(state):
        return path + [state]
    for child in generate_children(state):
        if child not in visited:
            visited.add(child)
            result = backtrack(child, path + [child], visited)
            visited.remove(child)
            if result:
                return result
    return None

def a_star(initial_state):
    frontier = []
    heapq.heappush(frontier, (heuristic(initial_state), 0, initial_state, []))
    visited = set()
    while frontier:
        _, cost, state, path = heapq.heappop(frontier)
        if is_goal(state):
            return path + [state]
        if state in visited:
            continue
        visited.add(state)
        for child in generate_children(state):
            if child not in visited:
                new_cost = cost + 1
                priority = new_cost + heuristic(child)
                heapq.heappush(frontier, (priority, new_cost, child, path + [state]))
    return None

initial_state = (frozenset(people), frozenset(), frozenset(), "left")

dfs_result = dfs(initial_state, [], set())
backtrack_result = backtrack(initial_state, [], set())
astar_result = a_star(initial_state)

print("Number of Steps (for 2 couples)")
print(f"DFS: {len(dfs_result)} steps")
print(f"Backtracking: {len(backtrack_result)} steps")
print(f"A*: {len(astar_result)} steps")

# Print the steps
def print_solution(name, result):
    print(f"\n{name} Solution ({len(result)} steps):")
    for i, (l, i_, r, b) in enumerate(result):
        print(f"Step {i}: L={sorted(l)}, I={sorted(i_)}, R={sorted(r)}, Boat={b}")

print_solution("DFS", dfs_result)
print_solution("Backtracking", backtrack_result)
print_solution("A*", astar_result)


Number of Steps (for 2 couples)
DFS: 17 steps
Backtracking: 19 steps
A*: 11 steps

DFS Solution (17 steps):
Step 0: L=['H2', 'W2'], I=['H1', 'W1'], R=[], Boat=island
Step 1: L=['H2', 'W2'], I=[], R=['H1', 'W1'], Boat=right
Step 2: L=['H2', 'W2'], I=['H1'], R=['W1'], Boat=island
Step 3: L=['H1', 'H2', 'W2'], I=[], R=['W1'], Boat=left
Step 4: L=['W2'], I=['H1', 'H2'], R=['W1'], Boat=island
Step 5: L=['W2'], I=[], R=['H1', 'H2', 'W1'], Boat=right
Step 6: L=['W2'], I=['H1', 'W1'], R=['H2'], Boat=island
Step 7: L=['W2'], I=['W1'], R=['H1', 'H2'], Boat=right
Step 8: L=['W2'], I=['H1', 'H2', 'W1'], R=[], Boat=island
Step 9: L=['H2', 'W2'], I=['H1', 'W1'], R=[], Boat=left
Step 10: L=[], I=['H1', 'H2', 'W1', 'W2'], R=[], Boat=island
Step 11: L=[], I=['H2', 'W2'], R=['H1', 'W1'], Boat=right
Step 12: L=[], I=['H1', 'H2', 'W2'], R=['W1'], Boat=island
Step 13: L=[], I=['W2'], R=['H1', 'H2', 'W1'], Boat=right
Step 14: L=[], I=['H2', 'W2'], R=['H1', 'W1'], Boat=island
Step 15: L=[], I=[], R=['H1', 'H